In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
# grok_api_key = os.getenv('GROK_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"Deepseek API Key exists and begins {deepseek_api_key[:4]}")
else:
    print("Deepseek API Key not set (and this is optional)")

In [ ]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url= "https://api.deepseek.com"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)

In [ ]:
OPENAI_MODEL = "gpt-5.5"
CLAUDE_MODEL = "claude-opus-4-7"
DEEPSEEK_MODEL = "deepseek-v4-pro"
GEMINI_MODEL = "gemini-3.5-flash"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-haiku-4-5"
# DEEPSEEK_MODEL = "deepseek-v4-flash"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [ ]:
run_python(pi)

## Now trying conversion to Julia

Code written to text file which can be copied to notebook cell to run - file name has model name too


In [ ]:
system_prompt = """
Your task is to convert Python code into high performance Julia code.
Respond only with Julia code. Do not provide any explanation other than occasional comments.
The Julia response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to Julia with the fastest possible implementation that produces identical output in the least time.
Although you may recognize that the code is creating an estimate of pi, you should not use pi in your code.

Your response will be written to a file called model-julia.jl in the current directory.
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def write_output(julia, model):
    with open(f"{model}-julia.jl", "w", encoding="utf-8") as f:
        f.write(julia)

In [ ]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```julia','').replace('```','')
    write_output(reply, model)
    

In [ ]:
port(deepseek, DEEPSEEK_MODEL, pi)

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)

In [ ]:
port(gemini, GEMINI_MODEL, pi)

In [ ]:
port(openai, OPENAI_MODEL, pi)